In [0]:

# Chennai Urban Intelligence Project
# Bronze Data Ingestion Pipeline

from pyspark.sql.functions import (
    col,
    current_timestamp,
    input_file_name,
    lit
)

from pyspark.sql.types import *

print("PySpark Bronze ingestion started")
print("Spark Version:", spark.version)


In [0]:

# Define Databricks storage paths

raw_path = "/Volumes/workspace/default/chennai_raw_input/"

bronze_path = "/Volumes/workspace/default/chennai_bronze/"

print("Raw Path:", raw_path)

print("Bronze Path:", bronze_path)


In [0]:

# List all source files in Raw volume

files = dbutils.fs.ls(raw_path)

for file in files:

    print(
        "File:", file.name,
        "| Size:", file.size, "bytes"
    )


In [0]:

# Read historical rainfall CSV

rainfall_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(raw_path + "01_historical_rainfall.csv")
)

print("Rainfall records:", rainfall_bronze.count())

print("Rainfall columns:", rainfall_bronze.columns)

rainfall_bronze.printSchema()

display(rainfall_bronze.limit(10))


In [0]:

# Add Bronze ingestion metadata

rainfall_bronze = (
    rainfall_bronze
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "source_file",
        lit("01_historical_rainfall.csv")
    )
)

display(rainfall_bronze.limit(10))


In [0]:
rainfall_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_rainfall")

print("Bronze rainfall table created successfully!")

In [0]:
bronze_rainfall = spark.table(
    "workspace.default.bronze_rainfall"
)

print("Bronze rainfall records:", bronze_rainfall.count())

bronze_rainfall.printSchema()

display(bronze_rainfall.limit(10))

In [0]:

from pyspark.sql.functions import current_timestamp, lit

# List of remaining CSV datasets
csv_datasets = {
    "bronze_flood": "02_flood_affected_areas.csv",
    "bronze_road_complaints": "03_road_damage_complaints.csv",
    "bronze_waterlogging": "06_waterlogging_locations.csv"
}

# Read and ingest each CSV dataset
for table_name, file_name in csv_datasets.items():

    print("\nProcessing:", file_name)

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(raw_path + file_name)
    )

    # Add ingestion metadata
    df = (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(file_name))
    )

    # Save as Bronze Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default." + table_name)
    )

    print("Table created:", table_name)
    print("Total records:", df.count())
    print("Columns:", df.columns)


In [0]:

# Step 1: Read traffic JSON file

traffic_bronze = (
    spark.read
    .option("multiLine", True)
    .json(raw_path + "04_traffic_congestion.json")
)

# Step 2: Check total records

print("Total traffic records:", traffic_bronze.count())

# Step 3: Check column names

print("Traffic columns:", traffic_bronze.columns)

# Step 4: Check schema

traffic_bronze.printSchema()

# Step 5: Display sample records

display(traffic_bronze.limit(10))


In [0]:

# Read emergency response JSON

emergency_bronze = (
    spark.read
    .option("multiLine", True)
    .json(raw_path + "08_emergency_response_rescue.json")
)

# Check record count
print("Total emergency records:", emergency_bronze.count())

# Check columns
print("Emergency columns:", emergency_bronze.columns)

# Check schema
emergency_bronze.printSchema()

# Display sample records
display(emergency_bronze.limit(10))


In [0]:

from pyspark.sql.functions import current_timestamp, lit

# Add ingestion metadata
emergency_bronze = (
    emergency_bronze
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn(
        "source_file",
        lit("08_emergency_response_rescue.json")
    )
)

# Save as Bronze Delta table
(
    emergency_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_emergency_response")
)

# Verify saved Bronze table
emergency_check = spark.table(
    "workspace.default.bronze_emergency_response"
)

print("Emergency Bronze records:", emergency_check.count())
print("Emergency Bronze columns:", emergency_check.columns)

emergency_check.printSchema()

display(emergency_check.limit(10))


In [0]:

# List all Bronze tables created so far

bronze_tables = [
    "bronze_rainfall",
    "bronze_flood",
    "bronze_road_complaints",
    "bronze_waterlogging",
    "bronze_traffic",
    "bronze_emergency_response"
]

for table_name in bronze_tables:

    full_name = "workspace.default." + table_name

    if not spark.catalog.tableExists(full_name):
        print("Table:", table_name, "| Not created yet")
        continue

    df = spark.table(full_name)

    print(
        "Table:", table_name,
        "| Records:", df.count(),
        "| Columns:", len(df.columns)
    )


In [0]:

# Read traffic JSON file

traffic_df = (
    spark.read
    .option("multiLine", True)
    .json(raw_path + "04_traffic_congestion.json")
)

# Add ingestion information

from pyspark.sql.functions import current_timestamp, lit

traffic_df = (
    traffic_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("04_traffic_congestion.json"))
)

# Save as Bronze Delta table

traffic_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_traffic"
)

print("Traffic Bronze table created successfully!")
print("Total records:", traffic_df.count())


In [0]:
%pip install openpyxl

In [0]:

# Step 1: Import pandas

import pandas as pd

# Step 2: Read Excel file

grievances_path = raw_path + "05_public_grievances.xlsx"

grievances_pd = pd.read_excel(grievances_path)

# Step 3: Check records

print("Total records:", len(grievances_pd))

# Step 4: Check columns

print("Column names:", grievances_pd.columns.tolist())

# Step 5: View sample records

display(grievances_pd.head(10))


In [0]:

# Convert Pandas DataFrame into PySpark DataFrame

grievances_spark = spark.createDataFrame(grievances_pd)

# Check total records
print("Total records:", grievances_spark.count())

# Check schema
grievances_spark.printSchema()

# Display sample records
display(grievances_spark.limit(10))


In [0]:

import pandas as pd

# Read population Excel file
population_path = raw_path + "07_population_density_by_zone.xlsx"

population_pd = pd.read_excel(population_path)

# Check total records
print("Total population records:", len(population_pd))

# Check column names
print("Population columns:", population_pd.columns.tolist())

# Display sample records
display(population_pd.head(10))


In [0]:

# Convert Pandas DataFrame into PySpark DataFrame

population_spark = spark.createDataFrame(population_pd)

# Check records
print("Total records:", population_spark.count())

# Check schema
population_spark.printSchema()

# Display sample records
display(population_spark.limit(10))


In [0]:

from pyspark.sql.functions import current_timestamp, lit

# Add ingestion information
population_spark = (
    population_spark
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("07_population_density_by_zone.xlsx"))
)

# Save as Bronze Delta table
population_spark.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_population"
)

# Verify table
population_check = spark.table("workspace.default.bronze_population")

print("Population Bronze table created!")
print("Total records:", population_check.count())

display(population_check.limit(10))
